In [1]:
import praw
from datetime import datetime
from modules.DocumentFactory import DocumentFactory, ArxivDocument
import urllib.parse
import xmltodict
import requests
import modules.Corpus as cps


In [2]:
reddit = praw.Reddit(
    client_id="9oP1sHTal7pwIWfhAXH2oA",
    client_secret="OqsgvIMBUHwG-VmKB66RDl9Jka7qUQ",
    user_agent="DataAcquisitionTD3"
)

In [3]:
# === Choose your topic (keyword) ===
topic = "coronavirus"

In [4]:
# === Create Corpus instance ===
corpus = cps.Corpus(nom=topic)
print(f"📦 Corpus créé: {corpus}")

📦 Corpus créé: Corpus: coronavirus
  - Nombre de documents: 0
  - Nombre d'auteurs: 0
  - ID du prochain document: 0


In [5]:
# documents = {}  # list of Document objects
# last_doc_id = 0
#
# # === Initialize authors dictionary ===
# authors = {}

In [6]:


# === Get Reddit posts ===
for submission in reddit.subreddit("all").search(topic, limit=10):
    text = submission.selftext.replace("\n", " ").strip()
    date_time = datetime.fromtimestamp(submission.created_utc)

    # Get author name
    author_name = submission.author.name if submission.author else "Unknown"

    # # Check if author already exists in dictionary
    # if author_name not in authors:
    #     # Create new Author instance
    #     authors[author_name] = auth.Author(name=author_name)

    document = DocumentFactory.create_document(
        source="Reddit",
        titre=submission.title,
        auteur=author_name,
        date=date_time,
        texte=text,
        url=submission.url,
        num_comments=submission.num_comments,
        score=submission.score,
        subreddit=submission.subreddit.display_name
    )
    # Add document to corpus
    corpus.add_document(document)

    # # Add document to author's production
    # authors[author_name].add_author_doc(document)
    #
    # documents[last_doc_id] = document
    # last_doc_id += 1

print(f"✅ Fetched corpus {corpus} Reddit documents.")
# print(f"✅ Fetched from documents {len(documents)} Reddit documents.")

✅ Fetched corpus Corpus: coronavirus
  - Nombre de documents: 10
  - Nombre d'auteurs: 10
  - ID du prochain document: 10 Reddit documents.


In [7]:


# Properly encode the query parameter to handle spaces and special characters
encoded_query = urllib.parse.quote(topic)
url = f"http://export.arxiv.org/api/query?search_query=all:{encoded_query}&start=0&max_results=10"

# === Fetch and parse XML ===
response = requests.get(url, timeout=30)
response.raise_for_status()
parsed = xmltodict.parse(response.text)

for entry in parsed["feed"]["entry"]:
    date_time = datetime.strptime(entry["published"], "%Y-%m-%dT%H:%M:%SZ")
    try:
        # Extract author name
        author = [aut['name'] for aut in entry['author']][0]
        #i = entry.get("author", "Unknown")
    except:
        author = entry['author']['name']

    co_author = [aut['name'] for aut in entry['author']] if isinstance(entry['author'], list) else []

    # Check if author already exists in dictionary
    # if auteur not in authors:
    #     # Create new Author instance
    #     authors[auteur] = auth.Author(name=auteur)


    document = DocumentFactory.create_document(
            source="Arxiv",
            titre=entry["title"],
            auteur=author,
            date=date_time,
            texte=entry["summary"],
            url=entry["id"],
            arxiv_id=entry["id"],
            co_auteurs=co_author,
            categories=entry.get("category", []),

        )

    # Add document to author's production
    # authors[auteur].add_author_doc(document)
    #
    # Add document to corpus
    corpus.add_document(document)

        # documents[last_doc_id] = document
        # last_doc_id += 1


print(f"✅ Fetched corpus {corpus} Arxiv documents.")

✅ Fetched corpus Corpus: coronavirus
  - Nombre de documents: 20
  - Nombre d'auteurs: 20
  - ID du prochain document: 20 Arxiv documents.


In [8]:
# Sauvegarder en format pickle
corpus.save()

✅ Corpus sauvegardé en format pickle: data/coronavirus_corpus.pkl


In [9]:
# Charger depuis pickle
loaded_pickle = cps.Corpus.load('data/coronavirus_corpus.pkl')

✅ Corpus chargé depuis: data/coronavirus_corpus.pkl


In [10]:
# 3.1 Afficher la taille du corpus (nombre de documents)
print(f"3.1 - Taille du corpus: {len(corpus.documents)} documents")

# 3.2 Pour chaque document, afficher le nombre de mots et de phrases
print("\n3.2 - Nombre de mots et de phrases par document:")
print("-" * 80)

for doc_id, document in corpus.documents.items():
    # Compter les mots (séparés par des espaces)
    nb_mots = len(document.texte.split())

    # Compter les phrases (séparées par des points)
    nb_phrases = len(document.texte.split('.'))

    print(f"Document {doc_id}: '{document.titre[:50]}...'")
    print(f"  - Nombre de mots: {nb_mots}")
    print(f"  - Nombre de phrases: {nb_phrases}")
    print()

# 3.3 Supprimer les documents trop petits (moins de 100 caractères)
print("\n3.3 - Suppression des documents trop petits (< 100 caractères)")
print(f"Nombre de documents avant suppression: {len(corpus.documents)}")

# Créer une liste des IDs à supprimer
docs_to_remove = []
for doc_id, document in corpus.documents.items():
    if len(document.texte) < 100:
        docs_to_remove.append(doc_id)
        print(f"  - Document {doc_id} supprimé: '{document.titre[:50]}...' (taille: {len(document.texte)} caractères)")

# Supprimer les documents
for doc_id in docs_to_remove:
    del corpus.documents[doc_id]

print(f"Nombre de documents après suppression: {len(corpus.documents)}")

# 3.4 Créer une unique chaîne de caractères contenant tous les documents
print("\n3.4 - Création d'une chaîne unique contenant tous les documents")

# Extraire tous les textes des documents
textes = [document.texte for document in corpus.documents.values()]

# Joindre tous les textes avec un espace
corpus_texte_complet = " ".join(textes)

print(f"Longueur totale du texte: {len(corpus_texte_complet)} caractères")
print(f"Nombre total de mots: {len(corpus_texte_complet.split())}")
print(f"\nAperçu des 200 premiers caractères:")
print(corpus_texte_complet[:200] + "...")

3.1 - Taille du corpus: 20 documents

3.2 - Nombre de mots et de phrases par document:
--------------------------------------------------------------------------------
Document 0: 'Made a Coronavirus Margarita...'
  - Nombre de mots: 0
  - Nombre de phrases: 1

Document 1: 'Covid surges across US after holidays amid low boo...'
  - Nombre de mots: 0
  - Nombre de phrases: 1

Document 2: 'Child poverty in the US jumped and income declined...'
  - Nombre de mots: 0
  - Nombre de phrases: 1

Document 3: 'A few glimpses from my visit to North Korea, befor...'
  - Nombre de mots: 0
  - Nombre de phrases: 1

Document 4: '[OC] Obesity rate in the United States (1985–2021)...'
  - Nombre de mots: 0
  - Nombre de phrases: 1

Document 5: 'On this day in 2020: The world lost the coronaviru...'
  - Nombre de mots: 0
  - Nombre de phrases: 1

Document 6: 'Queen tests positive for coronavirus, Buckingham P...'
  - Nombre de mots: 0
  - Nombre de phrases: 1

Document 7: 'UK government wins £122m pand

In [11]:
# Afficher les 7 premiers documents triés par titre (ordre inverse)
corpus.show_sorted_by_title()


Documents triés par titre ('A-Z')
Affichage de 10 document(s)

[ID: 10]
Titre: A Comparative Genomic Analysis of Coronavirus Families Using Chaos Game
  Representation and Fisher-Shannon Complexity
Auteur: S. K. Laha
Date: 2021-07-13 11:12:22
Texte: From its first emergence in Wuhan, China in December, 2019 the COVID-19
pandemic has caused unpreced...
--------------------------------------------------------------------------------

[ID: 19]
Titre: Coronavirus Geographic Dissemination at Chicago and its Potential
  Proximity to Public Commuter Rail
Auteur: Peter Fang
Date: 2020-03-24 13:37:06
Texte: The community spread of coronavirus at great Chicago area has severely
threatened the residents heal...
--------------------------------------------------------------------------------

[ID: 13]
Titre: Genomics-guided molecular maps of coronavirus targets in human cells: a
  path toward the repurposing of existing drugs to mitigate the pandemic
Auteur: Gennadi Glinsky
Date: 2020-03-30 17:49